In [ ]:
# ----------------- Step 1: Instalar paquetes necesarios -----------------

# ----------------- Step 2: Preparar datos y embeddings -----------------
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions

# Cargar datos
df = pd.read_parquet(r"C:\Users\manjo\Downloads\wiki_hybrid_chunks_300 (1).parquet")
texts = df['chunk_text'].astype(str).tolist()

# Crear embeddings
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(texts, show_progress_bar=True)

# ----------------- Step 3: Guardar en ChromaDB -----------------
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("wiki_chunks")

# Agregar documentos y embeddings a ChromaDB
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=[str(i) for i in range(len(texts))]
)

# ----------------- Step 4: Recuperar chunks relevantes -----------------
def retrieve_chunks(query, k=3):
    query_emb = embedder.encode([query])[0]
    results = collection.query(query_embeddings=[query_emb], n_results=k)
    return results['documents'][0]

# ----------------- Step 5: Construir prompt -----------------
def build_prompt(context, question):
    return f"""You are a helpful assistant. Only use the context below to answer the user's question clearly and concisely.

Context:
{context}

Question: {question}
Answer:"""

# ----------------- Step 6: Generar respuesta con HuggingFace Pipeline y fallback -----------------
from transformers import pipeline

# Puedes cambiar el modelo por otro local o de HuggingFace Hub
generator = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def generate_with_fallback(prompt, fallback="I'm sorry, I don't know the answer."):
    try:
        output = generator(prompt, max_new_tokens=256, temperature=0.7)
        answer = output[0]['generated_text'].split("Answer:")[-1].strip()
        if not answer:
            return fallback
        return answer
    except Exception as e:
        print("⚠️ Generation failed:", e)
        return fallback

# ----------------- Step 7: Ejemplo de uso -----------------
query = "What is the use of reinforcement learning in chatbots?"
chunks = retrieve_chunks(query, k=3)
context = " ".join(chunks)
prompt = build_prompt(context, query)
answer = generate_with_fallback(prompt)

print("📚 Retrieved Context:\n", context)
print("\n🤖 Answer:\n", answer)